# Predicting cortical response to product images — TRIBE v2Runs Meta's **TRIBE v2** brain-encoding model on four still images and compares the predictedcortical response to each.**Before you start: `Runtime → Change runtime type → GPU`.** This will not run on CPU inreasonable time.---### What this doesTRIBE v2 predicts fMRI responses to video, audio and text. Still images aren't a supported input,so we use the protocol from the paper's own in-silico experiments (§5.9): each image is **flashedfor 1 second** against a grey field, with a fixed gap between flashes, turning the set into a shortsilent video. Because there's no speech, this path skips both the transcription step and the**gated Llama-3.2-3B** dependency — no Hugging Face token needed.### What you get1. A **peri-stimulus time course** — a sanity check. If the response peaks around 5 s, that   reproduces Fig. 4A of the paper and confirms trial alignment is correct.2. **Network profiles** — which of 17 cortical networks each image drives, relative to the others.3. A **distinctiveness matrix** — how different each image's predicted signature is from the rest.4. **Cortical maps** for each image.### Read this before quoting any number- **No human looked at your images.** These are model predictions with no ground truth.- **No behavioural data exists in this model** — no purchase intent, recall, or preference. It  predicts a blood-oxygen signal, nothing about advertising performance.- **Objects are this model's weakest category.** In the paper, predicted-vs-measured agreement was  0.79 for places and **0.12 for tools and objects**. Product photography sits in that blind spot.- Predictions are for a **group average** of the study cohort, not your audience.- Weights are **CC-BY-NC-4.0**. Commercial use is restricted.Treat the output as a hypothesis generator, not a finding.

## 1 · Install

In [ ]:
!pip install -q "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git" 2>&1 | tail -2import torchprint("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU — switch runtime!")

## 2 · Upload your imagesRun the cell, then select **4 images** (PNG or JPG). Filenames become the labels in every chart,so name them something readable, e.g. `brass_engraved.png`.

In [ ]:
from google.colab import filesfrom pathlib import Pathimport shutilSTIM = Path("stimuli"); shutil.rmtree(STIM, ignore_errors=True); STIM.mkdir()up = files.upload()for name, data in up.items():    (STIM / name).write_bytes(data)paths = sorted(p for p in STIM.iterdir() if p.suffix.lower() in {".png",".jpg",".jpeg",".webp"})print(f"\n{len(paths)} images ready:")for p in paths: print("  ", p.stem)assert len(paths) >= 2, "upload at least 2 images"

## 3 · Build the stimulus videoPaper protocol: 1 s on, grey between, randomised order, repeated.`REPS = 2` shows each image twice in different sequence positions. TRIBE is **deterministic** —the same input always gives the same output — so repetitions don't average away noise the way theywould with a real scanner. What they control is **order effects**, which is the easiest objectionto a single-presentation result.`SOA = 8` (onset-to-onset seconds) matches the paper. Lower it to 6 to cut runtime by 25%; theresponse still peaks well inside the window.

In [ ]:
import numpy as np, random, jsonfrom PIL import Imageimport imageio.v2 as imageioREPS, SOA, ON_DUR, FPS, SIZE, GREY = 2, 8.0, 1.0, 16, (512,512), 128# FPS=16 matches V-JEPA-2's sampling of 64 frames per 4 s window.def load(p):    im = Image.open(p).convert("RGB"); im.thumbnail(SIZE, Image.LANCZOS)    canvas = Image.new("RGB", SIZE, (GREY,)*3)    canvas.paste(im, ((SIZE[0]-im.width)//2, (SIZE[1]-im.height)//2))    return np.asarray(canvas)frames_by_name = {p.stem: load(p) for p in paths}grey = np.full((SIZE[1], SIZE[0], 3), GREY, np.uint8)rng = random.Random(0); order = []for _ in range(REPS):    block = [p.stem for p in paths]; rng.shuffle(block); order += blockn_on, n_off = int(ON_DUR*FPS), int((SOA-ON_DUR)*FPS)frames, trials = [], []for i, name in enumerate(order):    trials.append({"index": i, "image": name, "onset": round(i*SOA, 3)})    frames += [frames_by_name[name]]*n_on + [grey]*n_offVIDEO = "stimulus.mp4"w = imageio.get_writer(VIDEO, fps=FPS, codec="libx264", macro_block_size=1,                       ffmpeg_params=["-pix_fmt","yuv420p"])for f in frames: w.append_data(f)w.close()DURATION = len(frames)/FPSprint(f"{VIDEO}: {DURATION:.0f}s, {len(trials)} trials, {len(frames_by_name)} images x {REPS} reps")print(f"~{int(DURATION*2)} clips to encode\n")for t in trials: print(f"  t={t['onset']:5.1f}s  {t['image']}")

## 4 · Load TRIBE v2`average_subjects` is set internally, so predictions come from the model's **"unseen subject"**pathway — the group-average response, which is what the paper validates for zero-shot use.

In [ ]:
from tribev2 import TribeModelDEV = "cuda" if torch.cuda.is_available() else "cpu"model = TribeModel.from_pretrained(    "facebook/tribev2",    cache_folder="./cache",    device=DEV,    config_update={        "data.video_feature.image.device": DEV,        "data.audio_feature.device": DEV,        "data.text_feature.device": DEV,    },)n = sum(p.numel() for p in model._model.parameters())print(f"TRIBE loaded: {n/1e6:.1f}M trainable params, TR={model.data.TR}s, device={DEV}")

## 5 · Run the modelThe slow part is **V-JEPA-2-giant** (1.03B params) encoding the video: every 0.5 s it processes thepreceding 4 s as **8,192 tokens**. Expect a few minutes on a GPU.`audio_only=True` is the important flag — it skips the word-transcription and text stages, which iswhat avoids the gated Llama model. Our stimulus is silent, so nothing is lost.

In [ ]:
import pandas as pd, timefrom tribev2.demo_utils import get_audio_and_text_eventsev = pd.DataFrame([{"type":"Video","filepath":VIDEO,"start":0,                    "timeline":"default","subject":"default"}])ev = get_audio_and_text_events(ev, audio_only=True)   # silent video -> no whisperX, no Llamat0 = time.time()preds, segments = model.predict(events=ev)print(f"\ndone in {time.time()-t0:.0f}s — preds {preds.shape} (timesteps, vertices)")times = np.array([s.start for s in segments], float)   # absolute time on the timelineo = np.argsort(times); preds, times = preds[o], times[o]

## 6 · Cortical atlasSchaefer-400 / 17 networks, native to the `fsaverage5` surface TRIBE predicts onto.Note the exclusion below: parcel 0 is `Background+FreeSurfer_Defined_Medial_Wall`, which isn't afunctional network. Leaving it in silently contaminates every average with 1,743 meaninglessvertices.

In [ ]:
import nibabel as nbimport urllib.requestBASE = ("https://github.com/ThomasYeoLab/CBIG/raw/master/stable_projects/brain_parcellation/"        "Schaefer2018_LocalGlobal/Parcellations/FreeSurfer5.3/fsaverage5/label")for h in ("lh", "rh"):    urllib.request.urlretrieve(        f"{BASE}/{h}.Schaefer2018_400Parcels_17Networks_order.annot",        f"{h}.Schaefer400.annot")labs, names = [], Nonefor h in ("lh","rh"):    lab, _, nm = nb.freesurfer.read_annot(f"{h}.Schaefer400.annot")    nm = [x.decode() for x in nm]    names = names or nm    labs.append(lab)full = np.concatenate(labs)net_of = {}for i, nm in enumerate(names):    bad = nm.startswith("Background") or "Medial_Wall" in nm      # not a network    parts = nm.split("_")    net_of[i] = "None" if bad else (parts[2] if len(parts) > 2 else "None")NETWORKS = sorted({v for v in net_of.values() if v != "None"})MASKS = {n: np.isin(full, [i for i,v in net_of.items() if v==n]) for n in NETWORKS}print(f"{len(NETWORKS)} networks, {sum(m.sum() for m in MASKS.values()):,} of {full.size:,} vertices "      f"({full.size - sum(m.sum() for m in MASKS.values()):,} medial wall excluded)")

## 7 · Align trials and compute contrastsFor each presentation we take the predicted response from onset to onset+`WIN` seconds, thenaverage across repetitions.The peak is found **empirically** rather than assumed. The README says predictions are alreadylag-corrected; the paper's §5.9 takes the response at t=5 s. Both can't be true of the same index,so we let the data decide — and if it lands near 5 s, that's independent confirmation the alignmentis right.Each image's contrast is *itself minus the mean of the others*, following the paper.

In [ ]:
WIN = int(SOA) - 1images = [p.stem for p in paths]per_img = {im: [] for im in images}for tr in trials:    idx = [int(np.argmin(np.abs(times-(tr["onset"]+dt)))) for dt in range(WIN+1)]    if abs(times[idx[0]] - tr["onset"]) > 1.5:   # onset not covered by predictions        continue    per_img[tr["image"]].append(preds[idx])curves = {im: np.mean(v, 0) for im, v in per_img.items() if v}assert curves, "no trials aligned — check SOA and stimulus duration"stack = np.stack([curves[im] for im in images])          # (images, time, vertices)peak  = int(np.argmax(np.abs(stack).mean(axis=(0,2))))at_peak  = stack[:, peak, :]contrast = np.stack([at_peak[i] - np.delete(at_peak, i, 0).mean(0) for i in range(len(images))])print(f"peak response at t = {peak}s after onset", "(≈5s expected)" if 3<=peak<=7 else "(!! unexpected)")for im in images:    print(f"  {im:28s} {len(per_img[im])} trials")

## 8 · Did it behave like a brain?

In [ ]:
import matplotlib.pyplot as pltplt.rcParams.update({"figure.dpi":130, "font.size":9, "axes.spines.top":False,                     "axes.spines.right":False})PAL = ["#2D6A9F", "#C3300A", "#3F8F5B", "#8A5AA8", "#B8860B", "#4A4A52"]fig, ax = plt.subplots(figsize=(6.2,3.2))for i, im in enumerate(images):    ax.plot(range(WIN+1), np.abs(curves[im]).mean(1), marker="o", ms=4,            color=PAL[i%len(PAL)], lw=2, label=im)ax.axvline(peak, color="#8A5600", ls="--", lw=1.2, zorder=0)ax.annotate(f"peak t={peak}s", (peak, ax.get_ylim()[1]), textcoords="offset points",            xytext=(5,-12), color="#8A5600", fontsize=8.5)ax.set_xlabel("seconds after image onset"); ax.set_ylabel("mean |predicted response|")ax.set_title("Peri-stimulus time course", loc="left", fontweight="bold")ax.legend(frameon=False, fontsize=8)plt.tight_layout(); plt.savefig("fig1_timecourse.png", bbox_inches="tight"); plt.show()print("A peak near 5s reproduces the haemodynamic lag in Fig. 4A of the paper.")

## 9 · Network profilesEach bar is one image's response in one network, **relative to the average of the other images**.Positive means that image drives the network more than its peers do.

In [ ]:
prof = np.array([[contrast[i][MASKS[n]].mean() for n in NETWORKS] for i in range(len(images))])ordr = np.argsort(-np.abs(prof).max(0))[:10]          # 10 most differentiating networkssel  = [NETWORKS[j] for j in ordr]fig, ax = plt.subplots(figsize=(7.2, 3.6))x = np.arange(len(sel)); w = 0.8/len(images)for i, im in enumerate(images):    ax.bar(x + i*w - 0.4 + w/2, prof[i][ordr], w*0.9, label=im,           color=PAL[i%len(PAL)], edgecolor="white", linewidth=0.6)ax.axhline(0, color="#4A4A52", lw=0.8)ax.set_xticks(x); ax.set_xticklabels(sel, rotation=35, ha="right", fontsize=8)ax.set_ylabel("contrast vs other images")ax.set_title("Predicted response by cortical network", loc="left", fontweight="bold")ax.legend(frameon=False, fontsize=8, ncol=2)plt.tight_layout(); plt.savefig("fig2_networks.png", bbox_inches="tight"); plt.show()print("VisCent/VisPeri = visual · DorsAttn = spatial attention · SalVentAttn = salience")print("Network names describe connectivity, not function. Don't read cognition off them.")

## 10 · How distinct is each image?Cosine distance between whole-brain contrast maps. **0** = indistinguishable predicted signature,higher = more distinct.This is the most defensible output here: "do these produce different responses" is a question aboutrepresentation, which the model genuinely answers — unlike "which performs better", which needsoutcome data the model has never seen.

In [ ]:
C = contrast - contrast.mean(1, keepdims=True)C = C / (np.linalg.norm(C, axis=1, keepdims=True) + 1e-9)D = 1 - C @ C.Tfig, ax = plt.subplots(figsize=(4.6,4.0))im_ = ax.imshow(D, cmap="pink_r", vmin=0)ax.set_xticks(range(len(images))); ax.set_yticks(range(len(images)))ax.set_xticklabels(images, rotation=35, ha="right", fontsize=8)ax.set_yticklabels(images, fontsize=8)for a in range(len(images)):    for b in range(len(images)):        ax.text(b, a, f"{D[a,b]:.2f}", ha="center", va="center", fontsize=8,                color="#17171B" if D[a,b] < D.max()*0.6 else "white")ax.set_title("Distinctiveness (cosine distance)", loc="left", fontweight="bold")plt.colorbar(im_, fraction=0.046).outline.set_visible(False)plt.tight_layout(); plt.savefig("fig3_distinctiveness.png", bbox_inches="tight"); plt.show()iu = np.triu_indices(len(images), 1)pairs = sorted(zip(D[iu], [(images[a],images[b]) for a,b in zip(*iu)]), reverse=True)print(f"most distinct : {pairs[0][1][0]} vs {pairs[0][1][1]}  ({pairs[0][0]:.3f})")print(f"most alike    : {pairs[-1][1][0]} vs {pairs[-1][1][1]}  ({pairs[-1][0]:.3f})")

## 11 · Cortical maps

In [ ]:
from nilearn import datasets, plottingfs = datasets.fetch_surf_fsaverage("fsaverage5")NV = 10242vmax = float(np.percentile(np.abs(contrast), 99))fig, axes = plt.subplots(len(images), 2, figsize=(7.0, 2.1*len(images)),                         subplot_kw={"projection":"3d"})axes = np.atleast_2d(axes)for i, im in enumerate(images):    for j, (hemi, mesh, bg) in enumerate(            [("left", fs.infl_left, fs.sulc_left), ("right", fs.infl_right, fs.sulc_right)]):        dat = contrast[i][:NV] if hemi == "left" else contrast[i][NV:]        plotting.plot_surf_stat_map(mesh, dat, hemi=hemi, bg_map=bg, view="lateral",                                    threshold=vmax*0.25, vmax=vmax, cmap="cold_hot",                                    colorbar=False, axes=axes[i,j], figure=fig)        if j == 0:            axes[i,j].text2D(-0.06, 0.5, im, transform=axes[i,j].transAxes, rotation=90,                             va="center", ha="center", fontsize=9, fontweight="bold")plt.tight_layout(); plt.savefig("fig4_cortical_maps.png", bbox_inches="tight", dpi=150); plt.show()print("Red = more response than the other images · Blue = less. Lateral view, both hemispheres.")

## 12 · Download everything

In [ ]:
summary = {    "images": images, "reps": REPS, "soa_s": SOA, "peak_s": peak,    "trials_per_image": {im: len(v) for im, v in per_img.items()},    "networks": NETWORKS,    "network_contrast": {im: {n: float(prof[i][j]) for j, n in enumerate(NETWORKS)}                         for i, im in enumerate(images)},    "distinctiveness": {f"{images[a]}|{images[b]}": float(D[a,b])                        for a in range(len(images)) for b in range(a+1, len(images))},    "caveats": [        "Model predictions only — no human was scanned viewing these images.",        "No behavioural data in the model: no purchase intent, recall or preference.",        "Objects were the weakest validated category in the paper (r=0.12 vs 0.79 for places).",        "Group-average prediction from the study cohort, not a specific audience.",        "Weights are CC-BY-NC-4.0 — commercial use restricted.",    ],}Path("results.json").write_text(json.dumps(summary, indent=2))np.save("contrast_maps.npy", contrast)import shutilshutil.make_archive("tribe_results", "zip", ".", base_dir=None)from google.colab import files as _ffor f in ["fig1_timecourse.png","fig2_networks.png","fig3_distinctiveness.png",          "fig4_cortical_maps.png","results.json"]:    _f.download(f)print("done")